# ADK: Enterprise Search Approaches — RAG, Discovery Engine & Agentic Search

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/adk_skills_discovery_engine_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/adk_skills_discovery_engine_demo.ipynb)

This notebook demonstrates **three distinct search approaches** an ADK agent can use, all managed through the **SkillToolset** (GCS-backed skill library). Each approach solves the same problem differently — helping partners advise customers on **when to use what**.

## The Question
> *"Find information about our data retention policies."*

| Approach | How It Works | Use When |
|----------|-------------|----------|
| **RAG (Vector Search)** | `AI.EMBED` + `VECTOR_SEARCH` + `AI.SIMILARITY` in BigQuery | You need precise semantic matching over your own curated corpus |
| **Discovery Engine Search** | Vertex AI Search — Google-managed indexing & ranking | Enterprise search at scale, no vector infra to maintain |
| **Agentic Search** | Agent uses `ExecuteBashTool` to explore systems autonomously | Open-ended exploration — agent decides where and how to search |

### March 2026 Features Demonstrated
- **GCS Skill Libraries** — Load agent skills from GCS ([v1.27.0](https://github.com/google/adk-python/releases/tag/v1.27.0))
- **ExecuteBashTool for Skills** — Governed bash execution with policy controls ([v1.27.0](https://github.com/google/adk-python/releases/tag/v1.27.0))
- **SkillToolset `additional_tools`** — Compose skills with real tools ([v1.27.0](https://github.com/google/adk-python/releases/tag/v1.27.0))
- **Discovery Engine: Structured Datastores + Regional Endpoints** ([v1.28.0](https://github.com/google/adk-python/releases/tag/v1.28.0))
- **BigQuery AI.EMBED + AI.SIMILARITY** — Native vector workflows (GA March 23, 2026)

### Requirements
- `google-adk >= 1.28.0`, `google-cloud-bigquery` installed.
- Vertex AI Search (Discovery Engine) API with a configured data store.
- BigQuery API and Vertex AI API enabled.

In [1]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" google-genai google-cloud-bigquery nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'iamtests-315719'  # @param {type:"string"}
location = 'us-central1'  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

Authenticated via Colab


### 2. [MANDATORY] Enable APIs

In [2]:
!gcloud services enable bigquery.googleapis.com aiplatform.googleapis.com discoveryengine.googleapis.com --project={project_id} --quiet
print("Success: BigQuery, Vertex AI, and Discovery Engine APIs enabled.")

Operation "operations/acat.p2-750496483448-14a240e5-83fa-4145-9ee1-93a9b840c9a4" finished successfully.
Success: BigQuery, Vertex AI, and Discovery Engine APIs enabled.


### 3. [PREREQUISITES] Prepare Sample Data

We create a document corpus in BigQuery that all three search approaches will query.

In [3]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=project_id, location=location)

dataset_id = f"{project_id}.search_demo"
dataset = bigquery.Dataset(dataset_id)
dataset.location = location
bq_client.create_dataset(dataset, exists_ok=True)

# Recreate table to avoid duplicate data on re-run
table_id = f"{dataset_id}.policy_documents"
schema = [
    bigquery.SchemaField("doc_id", "STRING"),
    bigquery.SchemaField("title", "STRING"),
    bigquery.SchemaField("content", "STRING"),
    bigquery.SchemaField("category", "STRING"),
    bigquery.SchemaField("last_updated", "DATE"),
]
bq_client.delete_table(table_id, not_found_ok=True)
table = bigquery.Table(table_id, schema=schema)
bq_client.create_table(table)

docs = [
    {"doc_id": "POL-001", "title": "Data Retention Policy v3.2",
     "content": "All customer PII must be retained for 7 years per GDPR Article 17. After the retention period, data must be securely deleted using DoD 5220.22-M standard. Backup copies follow the same retention schedule.",
     "category": "compliance", "last_updated": "2026-03-01"},
    {"doc_id": "POL-002", "title": "Cloud Storage Lifecycle Rules",
     "content": "Objects in the analytics-raw bucket transition to Nearline after 30 days and Coldline after 90 days. Retention locks are applied to compliance buckets with a 2555-day (7-year) hold.",
     "category": "infrastructure", "last_updated": "2026-02-15"},
    {"doc_id": "POL-003", "title": "BigQuery Dataset Expiration Standards",
     "content": "Production datasets must set default_table_expiration_ms to NULL (no auto-delete). Staging datasets expire after 72 hours. Temporary query result tables expire after 24 hours.",
     "category": "data-engineering", "last_updated": "2026-03-10"},
    {"doc_id": "POL-004", "title": "Employee Onboarding Handbook",
     "content": "New employees receive laptop, badge, and cloud access within 48 hours. All accounts follow least-privilege IAM policy.",
     "category": "hr", "last_updated": "2026-01-20"},
]

bq_client.insert_rows_json(table_id, docs)
print(f"Loaded {len(docs)} policy documents into {table_id}")

Loaded 4 policy documents into iamtests-315719.search_demo.policy_documents


### 4. Build the Skill Library

Each search approach is defined as a **Skill** — a reusable instruction template. In production, these would be loaded from GCS via `load_skill_from_gcs_dir()`. Here we define them inline for clarity.

In [6]:
from google.adk.skills.models import Skill, Frontmatter

# Skill 1: RAG via BigQuery Vector Search
rag_skill = Skill(
    frontmatter=Frontmatter(
        name="rag-vector-search",
        description="Search using RAG: generate embeddings with AI.EMBED, then find semantically similar documents using AI.SIMILARITY."
    ),
    instructions="""To perform RAG-based search:
1. Use AI.EMBED with the gemini-embedding-001 model to generate an embedding for the user's query.
2. Use AI.SIMILARITY to compare the query embedding against pre-computed document embeddings.
3. Return the top results ranked by similarity score.
4. Present results with: document title, similarity score, and a relevant excerpt.
USE THIS WHEN: The user needs precise semantic matching over a curated corpus with full control over the embedding pipeline."""
)

# Skill 2: Discovery Engine Search (Managed)
discovery_skill = Skill(
    frontmatter=Frontmatter(
        name="discovery-engine-search",
        description="Search using Google-managed Discovery Engine for enterprise-scale document retrieval with automatic ranking."
    ),
    instructions="""To perform Discovery Engine search:
1. Use the discovery_engine_search tool with a natural language query.
2. The tool handles indexing, ranking, and snippet extraction automatically.
3. Present results with: document title, relevant snippets, and source metadata.
USE THIS WHEN: The user wants managed enterprise search at scale without maintaining vector infrastructure."""
)

# Skill 3: Agentic Search via governed Bash
agentic_skill = Skill(
    frontmatter=Frontmatter(
        name="agentic-bash-search",
        description="Explore systems autonomously using governed bash commands to find, grep, and inspect files and logs."
    ),
    instructions="""To perform agentic search via bash:
1. Use the execute_bash tool to run search commands (grep, find, cat).
2. Start broad (find files matching a pattern), then narrow down (grep for specific content).
3. Iterate: if initial results are incomplete, refine your search based on what you found.
4. Present results with: file path, matching content, and your assessment.
USE THIS WHEN: The user needs open-ended exploration of local systems, logs, or config files where the data location is unknown."""
)

print(f"Skills defined: {rag_skill.frontmatter.name}, {discovery_skill.frontmatter.name}, {agentic_skill.frontmatter.name}")
print()
print("In production, load these from GCS:")
print("  from google.adk.skills import load_skill_from_gcs_dir")
print("  skill = load_skill_from_gcs_dir(bucket_name='my-skills', skill_id='rag-vector-search')")

Skills defined: rag-vector-search, discovery-engine-search, agentic-bash-search

In production, load these from GCS:
  from google.adk.skills import load_skill_from_gcs_dir
  skill = load_skill_from_gcs_dir(bucket_name='my-skills', skill_id='rag-vector-search')


### 5. Approach 1 — RAG via BigQuery Vector Search

The agent uses `AI.EMBED` to generate embeddings and `AI.SIMILARITY` for semantic matching — all in native BigQuery SQL. This is the **DIY approach** where you control the entire pipeline.

> **Use this when**: You need precise semantic matching over your own curated corpus, with full control over embeddings, chunking, and ranking.

In [10]:
!bq mk --connection --connection_type=CLOUD_RESOURCE --location=us-central1 --project_id iamtests-315719 vertex-ai-connection

Connection 750496483448.us-central1.vertex-ai-connection successfully created


In [14]:
!gcloud projects add-iam-policy-binding iamtests-315719 \
    --member=serviceAccount:bqcx-750496483448-z4zl@gcp-sa-bigquery-condel.iam.gserviceaccount.com \
    --role=roles/aiplatform.user

Updated IAM policy for project [iamtests-315719].
auditConfigs:
- auditLogConfigs:
  - logType: ADMIN_READ
  - logType: DATA_READ
  - logType: DATA_WRITE
  service: datastream.googleapis.com
- auditLogConfigs:
  - logType: ADMIN_READ
  - logType: DATA_READ
  - logType: DATA_WRITE
  service: storage.googleapis.com
- auditLogConfigs:
  - logType: ADMIN_READ
  - logType: DATA_READ
  - logType: DATA_WRITE
  service: pubsub.googleapis.com
- auditLogConfigs:
  - logType: ADMIN_READ
  - logType: DATA_READ
  - logType: DATA_WRITE
  service: aiplatform.googleapis.com
bindings:
- members:
  - serviceAccount:databoost-test@iamtests-315719.iam.gserviceaccount.com
  - user:jobigeor@gmail.com
  role: projects/iamtests-315719/roles/spanner_databoost_test
- members:
  - serviceAccount:750496483448-compute@developer.gserviceaccount.com
  - serviceAccount:ab-svc-1-workload-manager@iamtests-315719.iam.gserviceaccount.com
  - serviceAccount:ab-svc-2-db-compute-engine@iamtests-315719.iam.gserviceaccount.co

In [32]:
# Create a remote embedding model (requires Vertex AI Connection)
remote_model_sql = f"""
CREATE OR REPLACE MODEL `{dataset_id}.embed_model`
REMOTE WITH CONNECTION `{location}.vertex-ai-connection`
OPTIONS (ENDPOINT = 'gemini-embedding-001');
"""
print("Creating remote embedding model...")
try:
    bq_client.query(remote_model_sql).result()
    print("Success: Remote model created.")
except Exception as e:
    print(f"Note: {e}")
    print(f"Create a Vertex AI connection first:")
    print(f"  bq mk --connection --connection_type=CLOUD_RESOURCE --location={location} vertex-ai-connection")

Creating remote embedding model...
Success: Remote model created.


In [35]:
# Step 2: Generate embeddings for all documents
embed_sql = f"""
CREATE OR REPLACE TABLE `{dataset_id}.policy_embeddings` AS
SELECT
  doc_id, title, content, category,
  AI.EMBED(
    content,
    endpoint => 'text-embedding-005'
  ) AS embedding
FROM `{dataset_id}.policy_documents`;
"""
print("--- RAG Step 1: Generating embeddings with AI.EMBED ---")
try:
    job = bq_client.query(embed_sql)
    job.result()
    print(f"Embeddings generated. Rows: {job.num_dml_affected_rows}")
except Exception as e:
    print(f"Note: {e} (Requires remote model from previous cell)")

--- RAG Step 1: Generating embeddings with AI.EMBED ---
Embeddings generated. Rows: None


In [38]:
# Step 3: Semantic search — same question, vector similarity
search_query = "What is our data retention policy?"

similarity_sql = f"""
SELECT
  doc_id, title,
  AI.SIMILARITY(content, '{search_query}', endpoint => 'text-embedding-005') AS similarity_score,
  SUBSTR(content, 1, 120) AS excerpt
FROM `{dataset_id}.policy_embeddings`
ORDER BY similarity_score DESC
LIMIT 3;
"""

print(f"--- RAG Step 2: Semantic Search with AI.SIMILARITY ---")
print(f"Query: '{search_query}'\n")
try:
    results = bq_client.query(similarity_sql).to_dataframe()
    print(results.to_string(index=False))
except Exception as e:
    print(f"Note: {e} (Requires embeddings from previous cell)")

print("\n>> RAG gives you full control: you choose the model, chunking strategy, and ranking logic.")

--- RAG Step 2: Semantic Search with AI.SIMILARITY ---
Query: 'What is our data retention policy?'

 doc_id                                 title  similarity_score                                                                                                                  excerpt
POL-001            Data Retention Policy v3.2          0.739330 All customer PII must be retained for 7 years per GDPR Article 17. After the retention period, data must be securely del
POL-002         Cloud Storage Lifecycle Rules          0.639298 Objects in the analytics-raw bucket transition to Nearline after 30 days and Coldline after 90 days. Retention locks are
POL-003 BigQuery Dataset Expiration Standards          0.616224 Production datasets must set default_table_expiration_ms to NULL (no auto-delete). Staging datasets expire after 72 hour

>> RAG gives you full control: you choose the model, chunking strategy, and ranking logic.


### 6. Approach 2 — Discovery Engine Search (Google-Managed) - NOT WORKING

The agent uses `DiscoveryEngineSearchTool` — Google handles indexing, ranking, and snippet extraction. No vector infrastructure to manage.

> **Use this when**: Enterprise-scale search across documents and structured data. You want Google to handle the search infrastructure.

In [47]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.adk.tools.skill_toolset import SkillToolset
from google.adk.tools.discovery_engine_search_tool import DiscoveryEngineSearchTool
from google.genai import types
import os

# Configure your Discovery Engine data store
data_store_id = 'ny-taxi-test_1767036109471'  # @param {type:"string"}

# Initialize Discovery Engine Search (now supports structured datastores + regional endpoints)
search_tool = DiscoveryEngineSearchTool(
    data_store_id=f"projects/{project_id}/locations/{location}/collections/default_collection/dataStores/{data_store_id}",
    max_results=5
)
# Ensure the tool name matches what the skill expects
search_tool.name = "discovery_engine_search"

# Compose: SkillToolset with Discovery Engine as additional tool
discovery_toolset = SkillToolset(
    skills=[discovery_skill],
    additional_tools=[search_tool]
)

# Gemini 3.1 Pro Preview is only available in the 'global' location.
# Switch env var AFTER toolset init so regional tools keep using us-central1.
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

discovery_agent = Agent(
    model="gemini-3.1-pro-preview",
    name="DiscoverySearchAgent",
    instruction="You are an enterprise search agent. Use your Discovery Engine skill to find relevant documents.",
    tools=[discovery_toolset, search_tool]  # Explicitly adding search_tool here
)

runner = Runner(
    agent=discovery_agent,
    session_service=InMemorySessionService(),
    app_name="discovery_search_demo",
    auto_create_session=True
)

async def run_discovery_search():
    query = "How many taxis are there?"
    print(f"--- Discovery Engine Search ---")
    print(f"Query: '{query}'\n")

    message = types.Content(parts=[types.Part(text=query)], role='user')
    async for event in runner.run_async(
        user_id="partner_user",
        session_id="march_session",
        new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent: {part.text}")
                if part.function_call:
                    print(f"[SYSTEM]: Calling tool '{part.function_call.name}'")
    print("\n>> Discovery Engine handles indexing, ranking, and snippets. Zero vector infra to manage.")

await run_discovery_search()

--- Discovery Engine Search ---
Query: 'How many taxis are there?'

[SYSTEM]: Calling tool 'load_skill'
[SYSTEM]: Calling tool 'discovery_engine_search'
[SYSTEM]: Calling tool 'discovery_engine_search'
Agent: I apologize, but I am currently unable to retrieve the information for you. The enterprise search service is encountering a technical configuration issue (`RESOURCE_PROJECT_INVALID`), which prevents me from accessing the document database. 

Please contact your system administrator to ensure the Discovery Engine project is properly configured.

>> Discovery Engine handles indexing, ranking, and snippets. Zero vector infra to manage.


### 7. Approach 3 — Agentic Search via Governed Bash

The agent uses `ExecuteBashTool` with policy controls to autonomously explore local systems — grepping files, inspecting logs, and iterating on what it finds.

> **Use this when**: Open-ended exploration of systems, logs, or config files where data location is unknown. The agent decides its own search strategy.

In [43]:
import tempfile, pathlib
from google.adk.tools.bash_tool import ExecuteBashTool, BashToolPolicy

# Create sample policy files for the agent to explore
workspace = pathlib.Path(tempfile.mkdtemp(prefix="policy_docs_"))

policies = {
    "data_retention_v3.2.txt": "DATA RETENTION POLICY v3.2\nAll customer PII retained for 7 years (GDPR Art. 17).\nSecure deletion via DoD 5220.22-M.\nBackups follow same schedule.",
    "storage_lifecycle.txt": "CLOUD STORAGE LIFECYCLE\nanalytics-raw: Nearline after 30d, Coldline after 90d.\nCompliance buckets: 7-year retention lock.",
    "bq_expiration.txt": "BIGQUERY EXPIRATION STANDARDS\nProduction: no auto-delete. Staging: 72h. Temp tables: 24h.",
    "onboarding.txt": "EMPLOYEE ONBOARDING\nLaptop, badge, cloud access within 48h. Least-privilege IAM.",
}
for filename, content in policies.items():
    (workspace / filename).write_text(content)

print(f"Sample policy files created in: {workspace}")
print(f"Files: {', '.join(policies.keys())}")

Sample policy files created in: /tmp/policy_docs_8n38xgn8
Files: data_retention_v3.2.txt, storage_lifecycle.txt, bq_expiration.txt, onboarding.txt


In [57]:
from google.adk.features import FeatureName

bash_tool = ExecuteBashTool(
    workspace=workspace,
    policy=BashToolPolicy(
        allowed_command_prefixes=("grep", "find", "cat", "head", "ls", "wc")
    )
)

# Compose: SkillToolset with governed Bash as additional tool
agentic_toolset = SkillToolset(
    skills=[agentic_skill],
    additional_tools=[bash_tool]
)

# Gemini 3.1 Pro Preview requires 'global' (already set in cell above)
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

agentic_agent = Agent(
    model="gemini-3.1-pro-preview",
    name="AgenticExplorer",
    instruction="""You are a system explorer agent. Use your bash skill to search through local files and find relevant information.
    You can only use read-only commands (grep, find, cat, head, ls, wc). Start broad, then narrow down.""",
    tools=[agentic_toolset, bash_tool]
)

runner = Runner(
    agent=agentic_agent,
    session_service=InMemorySessionService(),
    app_name="agentic_search_demo",
    auto_create_session=True
)

async def run_agentic_search():
    query = "What is our data retention policy?"
    print(f"--- Agentic Search (Governed Bash) ---")
    print(f"Query: '{query}'")
    print(f"Workspace: {workspace}")
    print(f"Allowed commands: grep, find, cat, head, ls, wc\n")
    message = types.Content(parts=[types.Part(text=query)], role='user')
    async for event in runner.run_async(
        user_id="explorer_1", session_id="agentic_session", new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent: {part.text}")
                if part.function_call:
                    print(f"[SYSTEM]: Executing '{part.function_call.name}' → {part.function_call.args}")
    print("\n>> Agentic search: the agent decides its own search strategy. Policy controls ensure only read-only commands.")

await run_agentic_search()

--- Agentic Search (Governed Bash) ---
Query: 'What is our data retention policy?'
Workspace: /tmp/policy_docs_8n38xgn8
Allowed commands: grep, find, cat, head, ls, wc

[SYSTEM]: Executing 'load_skill' → {'name': 'agentic-bash-search'}
[SYSTEM]: Executing 'execute_bash' → {'command': 'grep -rnl -i "retention" .'}
[SYSTEM]: Executing 'adk_request_confirmation' → {'originalFunctionCall': {'id': 'adk-dc7a124a-4abb-4e94-a321-0bf097926c9f', 'args': {'command': 'grep -rnl -i "retention" .'}, 'name': 'execute_bash'}, 'toolConfirmation': {'hint': 'Please approve or reject the bash command: grep -rnl -i "retention" .', 'confirmed': False}}

>> Agentic search: the agent decides its own search strategy. Policy controls ensure only read-only commands.


### 8. Comparison: When to Use What

| Dimension | RAG (Vector Search) | Discovery Engine | Agentic Search (Bash) |
|-----------|-------------------|-----------------|----------------------|
| **Control** | Full — you manage embeddings, indexing, ranking | Managed — Google handles everything | Autonomous — agent decides strategy |
| **Setup Cost** | Medium — need remote model + embedding table | Low — configure data store in console | Low — just files/systems to search |
| **Precision** | High — semantic similarity with custom models | High — Google's ranking algorithms | Variable — depends on agent's exploration |
| **Scale** | BigQuery scale (petabytes) | Enterprise scale (automatic) | Local/system scale |
| **Best For** | Custom corpus, domain-specific embeddings | Enterprise doc search, structured data | System exploration, log analysis, debugging |
| **March 2026** | `AI.EMBED` + `AI.SIMILARITY` GA | Structured datastores + regional endpoints | `ExecuteBashTool` with policy controls |

### 9. [BONUS] GCS-Backed Skill Library

In production, skills are stored in GCS for centralized management across teams and agents.

In [58]:
from google.adk.skills import list_skills_in_gcs_dir, load_skill_from_gcs_dir

# Production pattern: load skills from a managed GCS bucket
#
# bucket_name = 'my-enterprise-skills'
# skills_path = 'skill-library/v2'
#
# # List all available skills
# available = list_skills_in_gcs_dir(
#     bucket_name=bucket_name,
#     skills_base_path=skills_path,
#     project_id=project_id
# )
# for skill_id, meta in available.items():
#     print(f"  {skill_id}: {meta.description}")
#
# # Load a specific skill
# search_skill = load_skill_from_gcs_dir(
#     bucket_name=bucket_name,
#     skill_id='rag_vector_search',
#     skills_base_path=skills_path,
#     project_id=project_id
# )

print("GCS Skill Library pattern:")
print("  gs://my-enterprise-skills/skill-library/v2/")
print("    ├── rag_vector_search/")
print("    │     └── skill.md  (frontmatter + instructions)")
print("    ├── discovery_engine_search/")
print("    │     └── skill.md")
print("    └── agentic_bash_search/")
print("          └── skill.md")
print()
print("Update a skill in GCS → all agents using it get the new behavior. No code deploys.")

GCS Skill Library pattern:
  gs://my-enterprise-skills/skill-library/v2/
    ├── rag_vector_search/
    │     └── skill.md  (frontmatter + instructions)
    ├── discovery_engine_search/
    │     └── skill.md
    └── agentic_bash_search/
          └── skill.md

Update a skill in GCS → all agents using it get the new behavior. No code deploys.


### 10. Things to remember or know
- **Three approaches, one question**: Partners should advise customers based on the tradeoffs — control vs. convenience, setup cost vs. scale, and precision vs. autonomy.
- **Skills as the orchestration layer**: Skills separate *how* the agent searches from *what tools* it uses. New search methods can be added as skills without changing agent code.
- **Governed bash**: `ExecuteBashTool` includes policy-based command filtering and human-in-the-loop confirmation — safe for production use.
- **GCS skill libraries**: Store skills in GCS to version, audit, and share agent capabilities across teams via IAM.
- **Discovery Engine structured datastores**: As of [v1.28.0](https://github.com/google/adk-python/releases/tag/v1.28.0), Discovery Engine supports structured data stores and regional endpoints for data residency.
- **BigQuery AI functions**: `AI.EMBED` and `AI.SIMILARITY` are GA as of March 23, 2026 — bringing the full RAG pipeline into native SQL.